# 03 · Which layers of the audio model carry the signal?

So far the audio embedding is the **last hidden layer** of WavLM (mean + std over time). In speech models the middle layers often hold more information about pronunciation and fluency than the last one.
This notebook:
1. extracts mean+std pooled embeddings from **every layer** (cached),
2. scores each layer with the same ridge model and the same repeated stratified CV as before,
3. builds a few combinations (best layer, average of the best 3, average of all, concatenation of the best 3),
4. writes the best one as `audio_wavlm.npy` into a copy of the feature cache (the original is kept as `audio_wavlm_last.npy`),
   so `final.ipynb` can be re-run unchanged on this notebook's output.

Nothing here touches labels except inside cross-validation folds. The test set is never used for choosing layers.

In [ ]:
# ===================== SETTINGS =====================
AUDIO_MODEL = "microsoft/wavlm-base-plus"   # try "microsoft/wavlm-large" or "facebook/wav2vec2-base" later
DATA_DIR    = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
WORK_DIR    = "/kaggle/working"
SCAN_SEEDS  = [42]            # quick scan over layers
FINAL_SEEDS = [42, 43, 44]    # candidates are scored with repeated CV

import os, sys, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="X does not have valid feature names")

def _find_repo():
    cands = [Path.cwd(), Path(WORK_DIR) / "shl-grammar-scoring"]
    for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):
        cands += list(Path("/kaggle/input").glob(pat))
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    raise FileNotFoundError("Cannot find the repo's src/ folder: attach the code dataset (Add Input).")
REPO_DIR = _find_repo(); sys.path.insert(0, str(REPO_DIR))

# copy the COMMITTED 01 outputs (features, transcripts) into this session's cache
CACHE = Path(WORK_DIR) / "cache"
hits = list(Path("/kaggle/input").rglob("features/meta.csv"))
assert hits, "Features not found: attach the COMMITTED 01_extract_features notebook (Add Input -> Your Work)."
SRC_CACHE = hits[0].parent.parent
if not (CACHE / "features" / "meta.csv").exists():
    shutil.copytree(SRC_CACHE / "features", CACHE / "features", dirs_exist_ok=True)
for fn in ("transcripts.csv", "segments.jsonl"):
    if (SRC_CACHE / fn).exists() and not (CACHE / fn).exists():
        shutil.copy(SRC_CACHE / fn, CACHE / fn)

from src.config import Config
cfg = Config(DATA_DIR=DATA_DIR, CACHE_DIR=str(CACHE), OUTPUT_DIR=WORK_DIR).apply_env().seed_everything()
print("repo:", REPO_DIR, "| features copied from:", SRC_CACHE, "| audio model:", AUDIO_MODEL)

## 1 · Rebuild the file list and the preprocessed audio (same steps and order as 01)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from src.data import load_competition
from src.audio import preprocess_all

fd = cfg.features_dir
meta_feat = pd.read_csv(fd / "meta.csv")
data = load_competition(cfg, verbose=False)
meta = pd.concat([data["train"], data["test"]], ignore_index=True)
assert list(zip(meta.split, meta.stem)) == list(zip(meta_feat.split, meta_feat.stem)), \
    "file order differs from the 01 feature cache: re-run 01"
meta = preprocess_all(meta, cfg)                 # 16 kHz mono, trimmed, loudness-normalised (about 2 minutes)
tr = (meta.split == "train").to_numpy(); te = ~tr
y = meta.loc[tr, "label"].to_numpy(float)
print("rows:", len(meta), "| train:", int(tr.sum()), "| test:", int(te.sum()))

## 2 · Extract mean+std pooled embeddings from every layer (cached)

In [ ]:
from src.utils import get_device, free_gpu, progress

def extract_layers(paths, model_name):
    """Returns array (n_files, n_layers + 1, 2 * hidden): mean and std over time of every hidden layer."""
    import soundfile as sf, torch
    from transformers import AutoFeatureExtractor, AutoModel
    device = get_device()
    fe = AutoFeatureExtractor.from_pretrained(model_name, **cfg.hf_kwargs())
    model = AutoModel.from_pretrained(model_name, **cfg.hf_kwargs()).to(device).eval()
    out = []
    for p in progress(list(paths), desc=f"layers ({model_name})"):
        try:
            wav, sr = sf.read(p, dtype="float32")
            inp = fe(wav, sampling_rate=sr, return_tensors="pt").input_values.to(device)
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                hs = model(inp, output_hidden_states=True).hidden_states          # tuple of (1, T, D)
            out.append(np.stack([torch.cat([h[0].float().mean(0), h[0].float().std(0)]).cpu().numpy() for h in hs]))
        except Exception as e:
            print("failed:", p, repr(e)[:100]); out.append(None)
    good = [o for o in out if o is not None]
    assert good, "no embedding could be computed"
    fill = np.mean(good, axis=0)
    del model
    return np.stack([o if o is not None else fill for o in out]).astype(np.float32)

layers_path = fd / ("audio_layers_" + AUDIO_MODEL.replace("/", "_") + ".npy")
if layers_path.exists():
    L = np.load(layers_path)
else:
    L = extract_layers(meta["cache_path"].tolist(), AUDIO_MODEL); np.save(layers_path, L); free_gpu()
assert L.shape[0] == len(meta)
print("layer embeddings:", L.shape, "= (files, layers incl. input embedding, mean+std dims)")

## 3 · Score every layer (ridge, stratified 5-fold CV on the training clips)

In [ ]:
from src.models import run_cv, make_ridge, stratify_labels
strat = stratify_labels(y, cfg.CV_SPLITS)

def score(X, name, seeds):
    s = run_cv(make_ridge, X[tr], y, strat, seeds, cfg.CV_SPLITS, name, verbose=False)["summary"]
    return s["rmse_overall"], s["pearson_overall"]

rows = []
for l in range(L.shape[1]):
    r, p = score(L[:, l, :], f"layer{l}", SCAN_SEEDS)
    rows.append(dict(layer=l, cv_rmse=round(r, 4), cv_pearson=round(p, 4)))
scan = pd.DataFrame(rows); display(scan.to_string(index=False))
last = scan.iloc[-1]
print(f"last layer (what the pipeline used so far): RMSE {last.cv_rmse} | Pearson {last.cv_pearson}")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(scan.layer, scan.cv_rmse, "o-"); ax.set(xlabel="layer (0 = input embedding)", ylabel="CV RMSE (lower is better)", title=f"Ridge on each layer: {AUDIO_MODEL}")
ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(Path(WORK_DIR) / "audio_layer_scan.png", dpi=120); plt.show()

## 4 · Combine layers, pick the best by repeated CV, and write it into the feature cache

In [ ]:
order = scan.sort_values("cv_rmse").layer.tolist()
top3 = order[:3]
cands = {
    "last_layer (baseline)": L[:, -1, :],
    f"best_layer ({order[0]})": L[:, order[0], :],
    f"avg_top3 {sorted(top3)}": L[:, top3, :].mean(1),
    "avg_all_layers": L[:, 1:, :].mean(1),
    f"concat_top3 {sorted(top3)}": np.concatenate([L[:, l, :] for l in sorted(top3)], axis=1),
}
res = []
for name, X in cands.items():
    r, p = score(X, name, FINAL_SEEDS)
    res.append(dict(candidate=name, dims=X.shape[1], cv_rmse=round(r, 4), cv_pearson=round(p, 4)))
res = pd.DataFrame(res); display(res.to_string(index=False))

best_name = res.sort_values("cv_rmse").iloc[0].candidate
best_X = cands[best_name].astype(np.float32)
print("chosen:", best_name)

if not (fd / "audio_wavlm_last.npy").exists():
    shutil.copy(fd / "audio_wavlm.npy", fd / "audio_wavlm_last.npy")     # keep the original
np.save(fd / "audio_wavlm.npy", best_X)                                  # final.ipynb reads this file name
(fd / "audio_wavlm_choice.txt").write_text(f"{AUDIO_MODEL} | {best_name}\n")
print("saved", fd / "audio_wavlm.npy", best_X.shape)
print(sorted(p.name for p in fd.iterdir()))

## 5 · What to do next
Attach **this** (committed) notebook, instead of the 01 notebook, as the input of `final.ipynb`: its Output contains the complete feature cache with the improved `audio_wavlm.npy`, plus `transcripts.csv`. Then run `final.ipynb` and submit.

*Notes.* Layers are chosen with cross-validation on training clips only. The test clips are never used for the choice. Choosing among about 17 candidates on the same CV is slightly optimistic, so the leaderboard remains the final check.